In [ ]:
import pandas as pd
import numpy as np
import cupy as cp
import os
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRFRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 统一配置接口
# ==========================================
INPUT_FILE = "P4_Cleaned_Dataset.csv"
OUTPUT_CSV = "P4_Nested_MOBO_Metrics_Fast.csv"
OPTUNA_TRIALS = 50  # [加速策略 1] 迭代次数降至 50，足以逼近帕累托前沿
# ==========================================

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100 
    
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    
    return {
        'R2': round(r2, 3), 'RMSE': round(rmse, 3), 'RRMSE(%)': round(rrmse, 3),
        'MAE': round(mae, 3), 'MAPE(%)': round(mape, 3), 'd-index': round(d_index, 3)
    }

def objective(trial, X_df, y_np, groups):
    # 1. 动态生成特征掩码
    active_features = []
    for col in X_df.columns:
        if trial.suggest_categorical(f'mask_{col}', [True, False]):
            active_features.append(col)
            
    if len(active_features) == 0:
        return float('inf'), len(X_df.columns)
        
    # 2. 超参数采样
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200, step=50),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.3, 0.9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'tree_method': 'gpu_hist',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # 3. 恢复内部留一交叉验证
    logo = LeaveOneGroupOut()
    mae_scores = [] 
    
    X_np_subset = X_df[active_features].values
    
    for train_idx, val_idx in logo.split(X_np_subset, y_np, groups):
        X_tr, X_val = X_np_subset[train_idx], X_np_subset[val_idx]
        y_tr, y_val = y_np[train_idx], y_np[val_idx]
        
        model = XGBRFRegressor(**params)
        model.fit(cp.array(X_tr), cp.array(y_tr))
        
        preds = cp.asnumpy(model.predict(cp.array(X_val)))
        mae_scores.append(mean_absolute_error(y_val, preds))
        
    return np.mean(mae_scores), len(active_features)

def run_nested_mobo(input_file, output_csv):
    if not os.path.exists(input_file):
        print(f"错误: 找不到文件 {input_file}。请检查路径。")
        return
        
    df = pd.read_csv(input_file)
    meta_cols = ['Year', 'Zone', 'latitude', 'longitude', 'yield']
    feature_cols = [col for col in df.columns if col not in meta_cols]
    
    years = sorted(df['Year'].unique())
    all_y_true, all_y_pred = [], []
    fold_results = []
    
    print(f"\n{'='*75}")
    print(f">>> 启动 MOBO 联合优化 (修改)")
    print(f"    初始特征维度: {len(feature_cols)}")
    print(f"    单年寻优次数: {OPTUNA_TRIALS}")
    print(f"{'='*75}")

    for test_year in years:
        train_df = df[df['Year'] != test_year]
        test_df = df[df['Year'] == test_year]
        
        X_train_df = train_df[feature_cols]
        y_train_np = train_df['yield'].values
        groups_train = train_df['Year'].values  # 用于内层信息隔离
        
        X_test_df = test_df[feature_cols]
        y_test_np = test_df['yield'].values
        
        print(f"    [Year {test_year}] 正在寻找 Pareto 最优解...", end="", flush=True)
        
        # 声明多目标：分别最小化 MAE 和 特征数量
        study = optuna.create_study(directions=['minimize', 'minimize'], sampler=TPESampler(seed=42))
        func = lambda trial: objective(trial, X_train_df, y_train_np, groups_train)
        study.optimize(func, n_trials=OPTUNA_TRIALS)
        
        # 从 Pareto 前沿中提取最佳试验
        # 策略：以预测精度为绝对第一优先级，选择内层评级 MAE 最低的解
        pareto_front = study.best_trials
        best_trial = sorted(pareto_front, key=lambda t: t.values[0])[0]
        
        # 解析最佳参数和被选中的特征
        best_params = {k: v for k, v in best_trial.params.items() if not k.startswith('mask_')}
        best_params.update({'tree_method': 'gpu_hist', 'random_state': 42, 'n_jobs': -1})
        
        selected_features = [col for col in feature_cols if best_trial.params.get(f'mask_{col}', False)]
        
        print(f" 完成.")
        print(f"      -> 内部评级 MAE: {best_trial.values[0]:.2f} | 选用特征数: {len(selected_features)}/{len(feature_cols)}")
        
        # 提取最终使用的特征矩阵
        X_train_np_final = X_train_df[selected_features].values
        X_test_np_final = X_test_df[selected_features].values
        
        # 最终模型拟合 (使用外层全量训练集)
        final_model = XGBRFRegressor(**best_params)
        final_model.fit(cp.array(X_train_np_final), cp.array(y_train_np))
        
        # 预测与记录
        y_pred = cp.asnumpy(final_model.predict(cp.array(X_test_np_final)))
        
        all_y_true.extend(y_test_np)
        all_y_pred.extend(y_pred)
        
        m = calculate_metrics(y_test_np, y_pred)
        m['Test_Year'] = str(test_year)
        m['Used_Features'] = len(selected_features)
        m['Best_Params'] = str({k: best_params[k] for k in ['max_depth', 'colsample_bynode', 'subsample']})
        fold_results.append(m)
        
        print(f"      -> [验证结果] RRMSE: {m['RRMSE(%)']:.2f}% | MAPE: {m['MAPE(%)']:.2f}%\n")

    # 全局指标计算 (Overall Pooled)
    global_metrics = calculate_metrics(all_y_true, all_y_pred)
    global_metrics['Test_Year'] = 'Overall_Pooled'
    global_metrics['Used_Features'] = 'N/A'
    global_metrics['Best_Params'] = 'N/A'
    fold_results.append(global_metrics)
    
    print(f"{'='*75}")
    print(f">>> 联合优化全局汇总")
    m = global_metrics
    print(f"    R2: {m['R2']:.3f} | RMSE: {m['RMSE']:.2f} | RRMSE: {m['RRMSE(%)']:.2f}%")
    print(f"    MAE: {m['MAE']:.2f} | MAPE: {m['MAPE(%)']:.2f}% | d-index: {m['d-index']:.3f}")
    print(f"{'='*75}")

    cols = ['Test_Year', 'Used_Features', 'R2', 'RRMSE(%)', 'd-index', 'MAPE(%)', 'RMSE', 'MAE', 'Best_Params']
    pd.DataFrame(fold_results)[cols].to_csv(output_csv, index=False)
    print(f"指标已保存至: {output_csv}")

if __name__ == "__main__":
    run_nested_mobo(INPUT_FILE, OUTPUT_CSV)


===========================================================================
>>> 启动 MOBO 联合优化 (RTX 4050 极速对齐版)
    初始特征维度: 100
    单年寻优次数: 50
===========================================================================
    [Year 2016] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 514.51 | 选用特征数: 37/100
      -> [验证结果] RRMSE: 13.43% | MAPE: 8.82%

    [Year 2017] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 518.65 | 选用特征数: 56/100
      -> [验证结果] RRMSE: 13.28% | MAPE: 11.10%

    [Year 2018] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 545.94 | 选用特征数: 44/100
      -> [验证结果] RRMSE: 11.58% | MAPE: 9.07%

    [Year 2019] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 546.05 | 选用特征数: 46/100
      -> [验证结果] RRMSE: 12.12% | MAPE: 9.36%

    [Year 2020] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 533.52 | 选用特征数: 44/100
      -> [验证结果] RRMSE: 11.30% | MAPE: 9.29%

    [Year 2021] 正在寻找 Pareto 最优解... 完成.
      -> 内部评级 MAE: 556.76 | 选用特征数: 38/100
      -> [验证结果] RRMSE: 11.04% | MAPE: 9.01%

===========================================================================
>>> 联合优化全局汇总
    R2: 0.345 | RMSE: 750.52 | RRMSE: 12.19%
    MAE: 533.98 | MAPE: 9.44% | d-index: 0.717
===========================================================================
指标已保存至: P4_Nested_MOBO_Metrics_Fast.csv